In [58]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_path = Path("bakersfield-real-estate-analytics/data/raw")
clean_path = Path("bakersfield-real-estate-analytics/data/cleaned")

list(raw_path.glob("*.csv"))

zhvi = pd.read_csv(
    raw_path / "City_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
)

zhvi.head()

zhvi[ 
    (zhvi["RegionName"] == "Bakersfield") &
    (zhvi["StateName"] == "CA")
    ]

id_cols = [
    "RegionID",
    "SizeRank",
    "RegionName",
    "RegionType",
    "StateName",
    "State",
    "Metro",
    "CountyName" ]

bakersfield_zhvi = zhvi [
    (zhvi["RegionName"] == "Bakersfield") &
    (zhvi["StateName"] == "CA")
].copy()

bakersfield_zhvi_long = bakersfield_zhvi.melt(
    id_vars=id_cols,
    var_name="Date",
    value_name="ZHVI"
)

bakersfield_zhvi_long.head()

bakersfield_zhvi_long["Date"] = pd.to_datetime(
    bakersfield_zhvi_long["Date"]
)

bakersfield_zhvi_long = bakersfield_zhvi_long[
bakersfield_zhvi_long["Date"] >= "2020-01-01"
].copy()


bakersfield_zhvi_long.tail()

bakersfield_zhvi_long = bakersfield_zhvi_long.rename(columns={
    "RegionName": "City",
    "StateName": "State",
    "CountyName": "County",
    "ZHVI": "Home_Value"
})

bakersfield_zhvi_clean = bakersfield_zhvi_long[
    ["City", "State", "County", "Date", "Home_Value"]
].copy()

bakersfield_zhvi_clean.head()

bakersfield_zhvi_clean.isna().sum()

bakersfield_zhvi_clean.to_csv(
    clean_path / "bakersfield_zhvi_clean.csv",
    index=False
)

(clean_path / "bakersfield_zhvi_clean.csv").exists()

def clean_zillow_weekly(file_name, metric_name):
    df = pd.read_csv(raw_path / file_name)

    bakersfield = df[
        df["RegionName"].str.contains("Bakersfield", case=False, na=False)
    ].copy()

    id_cols = [
        "RegionID",
        "SizeRank",
        "RegionName",
        "RegionType",
        "StateName"
    ]
    long_df = bakersfield.melt(
        id_vars=id_cols,
        var_name="Date",
        value_name=metric_name
    )
    long_df["Date"] = pd.to_datetime(long_df["Date"])
    long_df = long_df[
        long_df["Date"] >= "2020-01-01"
    ].copy()

    long_df = long_df[
        ["RegionName", "StateName", "Date", metric_name]
    ].rename(columns={
        "RegionName": "Metro",
        "StateName": "State"
    })
    return long_df

inventory = clean_zillow_weekly(

"Metro_invt_fs_uc_sfrcondo_sm_week.csv",
"Inventory"
)

inventory.head()
inventory.tail()
inventory.isna().sum()

#now Going to apply to all the differennt raw files
new_listings = clean_zillow_weekly(
    "Metro_new_listings_uc_sfrcondo_sm_week.csv",
    "New_Listings"
)

median_list_price = clean_zillow_weekly(
    "Metro_mlp_uc_sfrcondo_sm_week.csv" ,
    "Median_List_Price"
)

sales_count = clean_zillow_weekly(
    "Metro_sales_count_raw_uc_sfrcondo_sm_week.csv" ,
    "Sales_Count"
)
median_sale_price = clean_zillow_weekly(
    "Metro_median_sale_price_uc_sfrcondo_sm_week.csv",
    "Median_Sale_Price"
)
sale_to_list = clean_zillow_weekly(
    "Metro_median_sale_to_list_uc_sfrcondo_sm_week.csv",
    "Sale_to_List_Ratio"
)
sold_above_list = clean_zillow_weekly(
    "Metro_pct_sold_above_list_uc_sfrcondo_sm_week.csv",
    "Pct_Sold_Above_List"
)
days_pending = clean_zillow_weekly(
    "Metro_mean_doz_pending_uc_sfrcondo_sm_week.csv",
    "Days_to_Pending"
)
pct_price_cut = clean_zillow_weekly(
    "Metro_perc_listings_price_cut_uc_sfrcondo_sm_week.csv",
    "Pct_Listings_Price_Cut"
)
price_cut_amount = clean_zillow_weekly(
    "Metro_mean_listings_price_cut_amt_uc_sfrcondo_sm_week.csv",
    "Price_Cut_Amount"
)

median_sale_price.head()

datasets = {
    "inventory": inventory,
    "new_listings": new_listings,
    "median_list_price": median_list_price,
    "sales_count": sales_count,
    "median_sale_price": median_sale_price,
    "sale_to_list": sale_to_list,
    "sold_above_list": sold_above_list,
    "days_pending": days_pending,
    "pct_price_cut": pct_price_cut,
    "price_cut_amount": price_cut_amount
}

for name, df in datasets.items():
    print(name, df.shape, df.isna().sum().sum())


new_listings[new_listings["New_Listings"].isna()]
pct_price_cut[
    pct_price_cut["Pct_Listings_Price_Cut"].isna()
]

#Combining all the datasets
from functools import reduce

weekly_dfs = [
    inventory,
    new_listings,
    median_list_price,
    sales_count,
    median_sale_price,
    sale_to_list,
    sold_above_list,
    days_pending,
    pct_price_cut,
    price_cut_amount
]
bakersfield_weekly = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=["Metro", "State", "Date"],
        how="outer"
    ),
    weekly_dfs
)
bakersfield_weekly = bakersfield_weekly.sort_values("Date").reset_index(drop=True)

bakersfield_weekly.head()
bakersfield_weekly.shape
bakersfield_weekly.isna().sum()
bakersfield_weekly["Date"].duplicated().sum()

#saving the data 
bakersfield_weekly.to_csv(
    clean_path / "bakersfield_weekly_market_clean.csv",
    index=False
)
(clean_path / "bakersfield_weekly_market_clean.csv").exists()

bakersfield_weekly.dtypes

bakersfield_zhvi_clean.columns.tolist()

bakersfield_zhvi_clean = bakersfield_zhvi_long.iloc[:, ~bakersfield_zhvi_long.columns.duplicated()].copy()

bakersfield_zhvi_clean = bakersfield_zhvi_clean[
    ["City", "State", "County", "Date", "Home_Value"]
]

bakersfield_zhvi_clean.columns.tolist()

bakersfield_zhvi_clean.to_csv(
    clean_path / "bakersfield_zhvi_clean.csv",
    index=False
)

test = pd.read_csv(clean_path / "bakersfield_zhvi_clean.csv")

test.columns.tolist()

inventory (344, 4) 0
new_listings (344, 4) 9
median_list_price (344, 4) 0
sales_count (341, 4) 0
median_sale_price (341, 4) 0
sale_to_list (341, 4) 1
sold_above_list (341, 4) 0
days_pending (344, 4) 0
pct_price_cut (344, 4) 1
price_cut_amount (344, 4) 0


['City', 'State', 'County', 'Date', 'Home_Value']